# 🚀 MEM LLM Orchestrator — Live Interactive Demo
### Autonomous Adaptive GPU Orchestration, Dynamic Lane Switching & Zero-OOM Defense

Welcome to the interactive testbench for **MEM Orchestrator**!
This notebook lets you experience the autonomous memory governor and adaptive lane switching engine in action on Google Colab cloud GPUs (T4 / V100 / A100) or CPU.

**What you will see in this demo:**
1. **Dynamic Hardware Calibration:** Auto-detects available GPU VRAM and tunes throughput lanes.
2. **Adaptive Lane Switching:** Watches the model promote/demote batch size and gradient accumulation in real-time.
3. **Zero-OOM Chaos Resilience:** Injects synthetic VRAM spikes (+1.2 GB shocks) and watches the MEM Governor prevent out-of-memory crashes on the fly.
4. **Autoregressive Text Generation:** Generates text using the trained weights.

In [ ]:
#@title 1. Setup & Hardware Discovery
import os, sys, torch

print("=" * 65)
print("  MEM ORCHESTRATOR - HARDWARE DISCOVERY")
print("=" * 65)
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  GPU Detected: {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("  Running on CPU (GPU acceleration not active)")
print("=" * 65)

# Clone repository if running in Colab
if not os.path.exists("mem-llm-orchestrator") and not os.path.exists("runtime"):
    !git clone https://github.com/nobazzy/mem-llm-orchestrator.git
    %cd mem-llm-orchestrator/mem_v3
elif os.path.exists("mem-llm-orchestrator/mem_v3"):
    %cd mem-llm-orchestrator/mem_v3

!pip install -q transformers datasets accelerate truststore

### 2. Run Live Adaptive Training
Launch the adaptive training loop for 200 steps. Watch how throughput, loss, and memory allocation stabilize in real time.

In [ ]:
#@title Launch 200-step training with AdaptiveLaneRunner
!python scripts/demo_video_chaos_defense.py --steps 200 --model-preset medium_75m --no-shock

### 3. 💥 Stress Test: Chaos Shock Injection (Zero-OOM Defense)
Now we inject simulated memory shocks (+1200MB VRAM) during live training.
Watch how the **AdaptiveLaneRunner** instantly detects memory pressure, performs an **emergency demotion**, defragments VRAM, and **keeps the training alive** without throwing a `CUDA Out of Memory` exception!

In [ ]:
#@title Chaos defense demo: injects +1200MB VRAM shocks every 50 steps
!python scripts/demo_video_chaos_defense.py --steps 200 --model-preset medium_75m --shock-interval 50 --shock-duration 25 --shock-size-mb 1200

### 4. 🧠 Text Generation & Inference Test
Test text generation autoregressively using the model weights and custom prompts:

In [ ]:
#@title Generate Text
prompt = "Artificial intelligence and neural networks are" #@param {type:"string"}
temperature = 0.7 #@param {type:"slider", min:0.1, max:1.5, step:0.1}
max_tokens = 80 #@param {type:"integer"}

!python scripts/run_inference.py --prompt "{prompt}" --temperature {temperature} --max-tokens {max_tokens}